### Image Processing

In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
       # "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [4]:
import base64;

In [5]:
with open("images/prop7.png",'rb') as f:
    image_bytes=base64.standard_b64encode(f.read()).decode('utf-8')

messages=[]
add_user_message(messages,[
                 {
                     "type" :"image",
                     "source": {
                         "type":"base64",
                         "media_type": "image/png",
                         "data": image_bytes
                     }
                 },
                 {
                     "type": "text",
                     "text": prompt
                 }]
                 )

ans=chat(messages)

In [6]:
print(ans.content[0].text)

# Satellite Image Property Fire Risk Analysis

**1. Residence Identification:**
The primary residence is a light-colored (gray/tan) roofed structure located in the center-lower portion of the image, featuring an irregular L-shaped or cross-shaped footprint with what appears to be multiple roof sections, completely surrounded by dense forest vegetation.

**2. Tree Overhang Analysis:**
Multiple large tree canopies directly overhang substantial portions of the residence roof, with overhanging vegetation covering approximately 50-75% of the visible roof area, particularly dense on the northern and eastern sections of the structure.

**3. Fire Risk Assessment:**
The overhanging trees create multiple ember catch points across the roof surface and establish direct fuel continuity between the surrounding forest canopy and the structure, with branches appearing to touch or come very close to the roof in several locations, creating significant wildfire vulnerability.

**4. Defensible Space Ident

#### PDF

In [7]:
with open('earth.pdf','rb') as f:
    file_bytes=base64.standard_b64encode(f.read()).decode('utf-8')

messages=[]
add_user_message(messages,[
    {
        "type": "document",
        "source":{
            "type" : "base64",
            "media_type": "application/pdf",
            "data": file_bytes          
        },
        "title":"earth.pdf",
        "citations":{
            "enabled": True
        }
    },
    {
        "type": "text",
        "text": "How were earth's atmospehere created ?"
    }
])

In [8]:
ans=chat(messages)
print(ans.content[0].text)

Earth's atmosphere and oceans were formed by volcanic activity and outgassing.


In [24]:
ans

Message(id='msg_011CffyEUXMoVkWGnZyhtSCC', container=None, content=[TextBlock(citations=[CitationPageLocation(cited_text="[42]\r\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\r\n", document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", type='text'), TextBlock(citations=None, text=' ', type='text'), TextBlock(citations=[CitationPageLocation(cited_text='[43] Water vapor from\r\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\r\nand comets.\r\n', document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text='Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.', type='text')], model='claude-sonnet-4-5-20250929', ro

### Citation from a text 

In [9]:
article_text="""
Earth's atmosphere
and oceans were formed by
volcanic activity
and
outgassing
.
[
43
]
Water vapor fromthese sources
condensed
into the oceans, augmented by water and ice from asteroids,
protoplanets
,and
comets
.
[
44
]
Sufficient water to fill the oceans may have been on Earth since it formed.
[
45
]
In thismodel, atmospheric
greenhouse gases
kept the oceans from freezing when the newly forming Sun
hadonly 70%
of its
current luminosity
.
[
46
]
By
3.5 Ga
,
Earth's magnetic field
was established, which helpedprevent the atmosphere from being stripped away by the
solar wind
.
[
47
]
As the molten outer layer of Earth cooled it
formed
the first solid
crust
, which is thought to have been
mafic
in composition. The first
continental crust
, which was more
felsic
in composition, formed by thepartial melting of this mafic crust.
[
49
]
The presence of grains of the
mineral zircon of Hadean age
in
Eoarchean
sedimentary rocks
suggests that at least some felsic crust existed as early as
4.4 Ga
, only
140
Ma
after Earth's formation.
[
50
]
There are two main models of how this initial small volume ofcontinental crust evolved to reach its current abundance:
[
51
]
(1) a relatively steady growth up to thepresent day,
[
52
]
which is supported by the radiometric dating of continental crust globally and (2) aninitial rapid growth in the volume of continental crust during the
Archean
, forming the bulk of the
"""

In [10]:
messages=[]
add_user_message(messages,[
    {
        "type": "document",
        "source":{
            "type" : "text",
            "media_type": "text/plain",
            "data": article_text          
        },
        "title":"Earth Article",
        "citations":{
            "enabled": True
        }
    },
    {
        "type": "text",
        "text": "How were earth's atmospehere created ?"
    }
])

In [11]:
ans=chat(messages)
print(ans.content[0].text)

Earth's atmosphere and oceans were formed by volcanic activity and outgassing.


In [12]:
ans

Message(id='msg_011Cfh811DEYV1ndtkvX2HLv', container=None, content=[TextBlock(citations=[CitationCharLocation(cited_text="\nEarth's atmosphere\nand oceans were formed by\nvolcanic activity\nand\noutgassing\n.\n", document_index=0, document_title='Earth Article', end_char_index=81, file_id=None, start_char_index=0, type='char_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1123, output_tokens=35, output_tokens_details=None, server_tool_use=None, service_tier='standard'), diagnostics=None)